<a href="https://colab.research.google.com/github/oseiwusufranco/NUTSGRAPH-Implementation/blob/main/NUTSGRAPH_implementation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# 0) SETUP: compatible installs + speed flags
# ============================================================

!pip install --upgrade torch==2.2.0 torchvision==0.17.0 torchaudio==2.2.0 \
    --index-url https://download.pytorch.org/whl/cu121

!pip install pyg_lib torch_scatter torch_sparse torch_cluster torch_spline_conv \
    -f https://data.pyg.org/whl/torch-2.2.0+cu121.html

!pip install torch-geometric

!pip install "numpy<2"
import numpy as np

import torch, os, random, math
print("Torch:", torch.__version__, "CUDA:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

# Install a PyTorch Geometric stack that matches your torch build.
# If this cell fails, tell me your torch version string again and I'll adjust the URL.
TORCH = torch.__version__.split('+')[0]
CUDA  = "cu" + torch.version.cuda.replace(".", "") if torch.cuda.is_available() else "cpu"
print("Installing PyG wheels for:", TORCH, CUDA)



# Speed toggles
torch.backends.cudnn.benchmark = True
torch.set_float32_matmul_precision("high")  # A100 loves this

from torch import nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import SAGEConv
from torch_geometric.loader import LinkNeighborLoader

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score


In [ ]:

# ============================================================
# 1) IMPORTS + REPRODUCIBILITY
# ============================================================
import os
import gc
import math
import time
import copy
import json
import random
import warnings
from dataclasses import dataclass

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    average_precision_score,
    roc_curve,
    precision_recall_curve,
    confusion_matrix
)

from torch_geometric.data import Data
from torch_geometric.nn import SAGEConv
from torch_geometric.loader import LinkNeighborLoader

import pyro
import pyro.distributions as dist
from pyro.infer import Predictive
from pyro.infer.mcmc import MCMC, NUTS

import arviz as az
import umap

warnings.filterwarnings("ignore")

def seed_everything(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
# ============================================================
# 2) CONFIG
# ============================================================
CFG = {
    # --------------------------------------------------------
    # DATA
    # --------------------------------------------------------
    "CSV_PATH": " ",
    "TARGET_ROWS": None,

    # --------------------------------------------------------
    # SPLITS
    # --------------------------------------------------------
    "TEST_SIZE": 0.12,
    "VAL_SIZE": 0.08,
    "SEED": 42,

    # --------------------------------------------------------
    # GRAPH / FEATURES
    # --------------------------------------------------------
    "SRC_COL": "IPV4_SRC_ADDR",
    "DST_COL": "IPV4_DST_ADDR",
    "MAX_EDGE_FEATURES": 32,
    "APPEND_L2_NORM": True,
    "USE_FLOAT16_EDGE_ATTR": False,

    # --------------------------------------------------------
    # GNN
    # --------------------------------------------------------
    "GNN_HIDDEN": 128,
    "GNN_LAYERS": 2,
    "GNN_DROPOUT": 0.10,

    # --------------------------------------------------------
    # EDGE CLASSIFIER
    # --------------------------------------------------------
    "CLS_HIDDEN": 128,
    "CLS_DROPOUT": 0.25,

    # --------------------------------------------------------
    # LOADERS
    # --------------------------------------------------------
    "BATCH_SIZE": 4096,
    "NUM_NEIGHBORS": [10, 5],
    "NUM_NEIGHBORS_BAYES": [3, 3],

    # --------------------------------------------------------
    # TRAINING
    # --------------------------------------------------------
    "LR": 2e-3,
    "WEIGHT_DECAY": 1e-5,
    "EPOCHS": 100,
    "PATIENCE": 5,
    "LABEL_SMOOTHING": 0.01,
    "USE_AMP": True,

    # --------------------------------------------------------
    # MC DROPOUT
    # --------------------------------------------------------
    "MC_PASSES": 10,

    # --------------------------------------------------------
    # NUTS (IMPROVED SETTINGS)
    # --------------------------------------------------------
    # Increased subset and samples to improve NUTS performance
    "NUTS_SUBSET_EDGES": ,
    "NUTS_NUM_WARMUP": 300,
    "NUTS_NUM_SAMPLES": 200,
    "NUTS_NUM_CHAINS": ,
    "NUTS_TARGET_ACCEPT": 0.85,
    "NUTS_MAX_TREE_DEPTH": 7,
    "NUTS_JIT_COMPILE": False,

    # --------------------------------------------------------
    # CALIBRATION
    # --------------------------------------------------------
    "ECE_BINS": 10,
}

print(json.dumps(CFG, indent=2))

In [ ]:

# ============================================================
# 3) LOAD DATASET
# ============================================================
df = pd.read_csv(CFG["CSV_PATH"])
print("Raw shape:", df.shape)

# Optional downsampling for practicality
if CFG["TARGET_ROWS"] is not None and len(df) > CFG["TARGET_ROWS"]:
    df = df.sample(CFG["TARGET_ROWS"], random_state=CFG["SEED"]).reset_index(drop=True)
    print("After subsampling:", df.shape)

label_candidates = ["Label", "label", "Attack", "attack", "Target", "target"]
LABEL_COL = None
for c in df.columns:
    if c in label_candidates or c.lower() in [x.lower() for x in label_candidates]:
        LABEL_COL = c
        break

if LABEL_COL is None:
    raise ValueError("Could not find the label column automatically.")

print("Detected label column:", LABEL_COL)
print(df[LABEL_COL].value_counts(dropna=False).head())
display(df.head())


In [ ]:

# ============================================================
# 4) BASIC CLEANING + LABEL ENCODING
# ============================================================
SRC_COL = CFG["SRC_COL"]
DST_COL = CFG["DST_COL"]

if SRC_COL not in df.columns or DST_COL not in df.columns:
    raise ValueError(f"Expected source/destination columns '{SRC_COL}' and '{DST_COL}'.")

# Replace infs and fill NaNs
df = df.replace([np.inf, -np.inf], np.nan)
df = df.fillna(0)

# Encode labels to 0/1 (or 0..K-1 if needed)
le = LabelEncoder()
y_all = le.fit_transform(df[LABEL_COL].astype(str))
n_classes = int(len(le.classes_))
print("Encoded classes:", list(le.classes_))
print("n_classes:", n_classes)

if n_classes != 2:
    print("WARNING: Notebook is optimized for binary intrusion classification, but will still run for multi-class where possible.")


In [ ]:

# ============================================================
# 5) FEATURE SELECTION + SCALING
# ============================================================
exclude_cols = {SRC_COL, DST_COL, LABEL_COL}
numeric_cols = [
    c for c in df.columns
    if c not in exclude_cols and pd.api.types.is_numeric_dtype(df[c])
]

if len(numeric_cols) == 0:
    raise ValueError("No numeric columns found for flow features.")

# Keep top-K most variable numeric features
var_series = df[numeric_cols].var(numeric_only=True).sort_values(ascending=False)
selected_num_cols = var_series.index[:CFG["MAX_EDGE_FEATURES"]].tolist()
print("Selected numeric feature count:", len(selected_num_cols))
print(selected_num_cols[:10], "..." if len(selected_num_cols) > 10 else "")

scaler = StandardScaler()
X_num = scaler.fit_transform(df[selected_num_cols].astype(np.float32))

if CFG["APPEND_L2_NORM"]:
    l2 = np.linalg.norm(X_num, axis=1, keepdims=True)
    X_edge = np.concatenate([X_num, l2], axis=1).astype(np.float32)
else:
    X_edge = X_num.astype(np.float32)

print("Edge feature matrix shape:", X_edge.shape)


In [ ]:

# ============================================================
# 6) GRAPH BUILD (FAST, VECTORISED)
# ============================================================
src = df[SRC_COL].astype("string").to_numpy()
dst = df[DST_COL].astype("string").to_numpy()

all_endpoints = np.concatenate([src, dst], axis=0)
codes, uniques = pd.factorize(all_endpoints, sort=False)

E = len(src)
src_id = codes[:E].astype(np.int64, copy=False)
dst_id = codes[E:].astype(np.int64, copy=False)
N = len(uniques)

edge_index = torch.tensor(np.vstack([src_id, dst_id]), dtype=torch.long)
edge_attr = torch.tensor(X_edge, dtype=torch.float32)
edge_label = torch.tensor(y_all, dtype=torch.long)

print(f"Nodes: {N:,}")
print(f"Edges: {E:,}")
print("edge_index:", tuple(edge_index.shape))
print("edge_attr:", tuple(edge_attr.shape))
print("edge_label:", tuple(edge_label.shape))


In [ ]:

# ============================================================
# 7) NODE FEATURES = MEAN OF INCIDENT EDGE FEATURES
# ============================================================
# For each node, aggregate incoming and outgoing edge attributes.
# This matches the practical graph-construction spirit in the paper.

Fdim = edge_attr.size(1)
node_feat_sum = torch.zeros((N, Fdim), dtype=torch.float32)
node_deg = torch.zeros(N, dtype=torch.float32)

node_feat_sum.index_add_(0, torch.from_numpy(src_id), edge_attr)
node_feat_sum.index_add_(0, torch.from_numpy(dst_id), edge_attr)
node_deg.index_add_(0, torch.from_numpy(src_id), torch.ones(E))
node_deg.index_add_(0, torch.from_numpy(dst_id), torch.ones(E))

node_x = node_feat_sum / node_deg.clamp_min(1.0).unsqueeze(1)
print("node_x:", tuple(node_x.shape))


In [ ]:

# ============================================================
# 8) TRAIN / VAL / TEST SPLIT (EDGE-LEVEL STRATIFIED)
# ============================================================
all_eids = np.arange(E)

train_val_idx, test_idx = train_test_split(
    all_eids,
    test_size=CFG["TEST_SIZE"],
    random_state=CFG["SEED"],
    stratify=y_all
)

val_adjusted = CFG["VAL_SIZE"] / (1 - CFG["TEST_SIZE"])

train_idx, val_idx = train_test_split(
    train_val_idx,
    test_size=val_adjusted,
    random_state=CFG["SEED"],
    stratify=y_all[train_val_idx]
)

print("Train:", len(train_idx))
print("Val  :", len(val_idx))
print("Test :", len(test_idx))


In [ ]:
# ============================================================
# 9) BUILD PyG DATA OBJECT
# ============================================================
data = Data(
    x=node_x,
    edge_index=edge_index,
    edge_attr=edge_attr,
    edge_label=edge_label
)

data.train_eid = torch.tensor(train_idx, dtype=torch.long)
data.val_eid   = torch.tensor(val_idx, dtype=torch.long)
data.test_eid  = torch.tensor(test_idx, dtype=torch.long)

# Move all tensors within the data object to the appropriate device
data = data.to(device)

print(data)

In [ ]:
# ============================================================
# 10) MODEL DEFINITIONS + TEMP SCALING
# ============================================================
class GraphSAGEEncoder(nn.Module):
    def __init__(self, in_dim, hidden_dim, num_layers=2, dropout=0.1):
        super().__init__()
        self.convs = nn.ModuleList()
        self.convs.append(SAGEConv(in_dim, hidden_dim))
        for _ in range(num_layers - 1):
            self.convs.append(SAGEConv(hidden_dim, hidden_dim))
        self.dropout = dropout

    def forward(self, x, edge_index):
        for i, conv in enumerate(self.convs):
            x = conv(x, edge_index)
            x = F.relu(x)
            x = F.dropout(x, p=self.dropout, training=self.training)
        return x


class EdgeClassifier(nn.Module):
    def __init__(self, emb_dim, edge_dim, hidden_dim=128, dropout=0.25, out_dim=2):
        super().__init__()
        in_dim = emb_dim * 2 + edge_dim
        self.lin1 = nn.Linear(in_dim, hidden_dim)
        self.lin2 = nn.Linear(hidden_dim, out_dim)
        self.dropout = dropout

    def forward(self, src_emb, dst_emb, edge_attr):
        z = torch.cat([src_emb, dst_emb, edge_attr], dim=-1)
        z = self.lin1(z)
        z = F.relu(z)
        z = F.dropout(z, p=self.dropout, training=self.training)
        z = self.lin2(z)
        return z


class GraphEdgeModel(nn.Module):
    def __init__(self, node_in_dim, edge_dim, hidden_dim=128, gnn_layers=2,
                 gnn_dropout=0.1, cls_hidden=128, cls_dropout=0.25, out_dim=2):
        super().__init__()
        self.encoder = GraphSAGEEncoder(node_in_dim, hidden_dim, gnn_layers, gnn_dropout)
        self.classifier = EdgeClassifier(hidden_dim, edge_dim, cls_hidden, cls_dropout, out_dim)

    def forward(self, batch):
        x = self.encoder(batch.x, batch.edge_index)
        src, dst = batch.edge_label_index
        src_emb = x[src]
        dst_emb = x[dst]
        # Use batch.input_id to retrieve the edge attributes for the target links
        # from the global data.edge_attr
        edge_attr_for_labels = data.edge_attr[batch.input_id]
        logits = self.classifier(src_emb, dst_emb, edge_attr_for_labels)
        return logits

    @torch.no_grad()
    def full_node_embeddings(self, data_obj, device):
        self.eval()
        x = self.encoder(data_obj.x.to(device), data_obj.edge_index.to(device))
        return x.detach().cpu()


class ModelWithTemperature(nn.Module):
    """
    A thin decorator, which wraps a model with temperature scaling
    model (nn.Module):
        A classification neural network
        NB: Output of the neural network should be the classification logits,
            NOT the softmax (or log softmax)!
    """
    def __init__(self, model):
        super(ModelWithTemperature, self).__init__()
        self.model = model
        self.temperature = nn.Parameter(torch.ones(1) * 1.5)

    def forward(self, batch):
        logits = self.model(batch)
        return self.temperature_scale(logits)

    def temperature_scale(self, logits):
        """
        Perform temperature scaling on logits
        """
        # Expand temperature to match the size of logits
        temperature = self.temperature.unsqueeze(1).expand(logits.size(0), logits.size(1))
        return logits / temperature

    # This function probably should live outside of this class, but whatever
    def set_temperature(self, valid_loader, device):
        self.to(device)
        nll_criterion = nn.CrossEntropyLoss().to(device)
        ece_criterion = None # We optimize NLL

        # First: collect all the logits and labels for the validation set
        logits_list = []
        labels_list = []
        with torch.no_grad():
            for batch in valid_loader:
                batch = batch.to(device)
                logits = self.model(batch)
                logits_list.append(logits)
                labels_list.append(batch.edge_label)
            logits = torch.cat(logits_list).to(device)
            labels = torch.cat(labels_list).to(device)

        # Calculate NLL and ECE before temperature scaling
        before_temperature_nll = nll_criterion(logits, labels).item()
        print('Before temperature - NLL: %.3f' % (before_temperature_nll))

        # Next: optimize the temperature w.r.t. NLL
        optimizer = torch.optim.LBFGS([self.temperature], lr=0.01, max_iter=50)

        def eval():
            optimizer.zero_grad()
            loss = nll_criterion(self.temperature_scale(logits), labels)
            loss.backward()
            return loss
        optimizer.step(eval)

        # Calculate NLL and ECE after temperature scaling
        after_temperature_nll = nll_criterion(self.temperature_scale(logits), labels).item()
        print('Optimal temperature: %.3f' % self.temperature.item())
        print('After temperature - NLL: %.3f' % (after_temperature_nll))

        return self

In [ ]:
# ============================================================
# 11) DATALOADERS
# ============================================================
def make_loader(eids, shuffle, num_neighbors):
    edge_label_index = data.edge_index[:, eids]
    edge_label = data.edge_label[eids]

    return LinkNeighborLoader(
        data=data,
        num_neighbors=num_neighbors,
        batch_size=CFG["BATCH_SIZE"],
        edge_label_index=edge_label_index,
        edge_label=edge_label,
        input_id=eids, # Pass the original edge IDs for the target links
        shuffle=shuffle
    )

train_loader = make_loader(data.train_eid, shuffle=True,  num_neighbors=CFG["NUM_NEIGHBORS"])
val_loader   = make_loader(data.val_eid,   shuffle=False, num_neighbors=CFG["NUM_NEIGHBORS"])
test_loader  = make_loader(data.test_eid,  shuffle=False, num_neighbors=CFG["NUM_NEIGHBORS"])

print("Loaders ready.")

In [ ]:

# ============================================================
# 12) METRICS + UTILITIES
# ============================================================
def calculate_calibration_errors(y_true, probs_pos, n_bins=10):
    y_true = np.asarray(y_true).astype(int)
    probs_pos = np.asarray(probs_pos).astype(float)

    bins = np.linspace(0.0, 1.0, n_bins + 1)
    bin_ids = np.digitize(probs_pos, bins) - 1
    bin_ids = np.clip(bin_ids, 0, n_bins - 1)

    ece = 0.0
    mce = 0.0

    for b in range(n_bins):
        idx = bin_ids == b
        if idx.sum() == 0:
            continue
        conf = probs_pos[idx].mean()
        acc = y_true[idx].mean()
        gap = abs(acc - conf)
        ece += (idx.sum() / len(y_true)) * gap
        mce = max(mce, gap)
    return float(ece), float(mce)

def predictive_entropy(probs):
    probs = np.clip(probs, 1e-12, 1.0)
    return -(probs * np.log(probs)).sum(axis=1)

def fpr_at_95_tpr(y_true, score):
    fpr, tpr, _ = roc_curve(y_true, score)
    valid = np.where(tpr >= 0.95)[0]
    if len(valid) == 0:
        return np.nan
    return float(fpr[valid[0]])

def summarize_predictions(y_true, probs, prefix=""):
    y_pred = probs.argmax(axis=1)
    pos_probs = probs[:, 1] if probs.shape[1] > 1 else probs[:, 0]

    out = {}
    out[f"{prefix}Accuracy"] = accuracy_score(y_true, y_pred)
    out[f"{prefix}F1"] = f1_score(y_true, y_pred, average="macro", zero_division=0)
    out[f"{prefix}Precision"] = precision_score(y_true, y_pred, average="macro", zero_division=0)
    out[f"{prefix}Recall"] = recall_score(y_true, y_pred, average="macro", zero_division=0)

    try:
        if probs.shape[1] == 2:
            out[f"{prefix}ROC_AUC"] = roc_auc_score(y_true, pos_probs)
            out[f"{prefix}AUPR"] = average_precision_score(y_true, pos_probs)
        else:
            out[f"{prefix}ROC_AUC"] = roc_auc_score(y_true, probs, multi_class="ovr")
            out[f"{prefix}AUPR"] = np.nan
    except Exception:
        out[f"{prefix}ROC_AUC"] = np.nan
        out[f"{prefix}AUPR"] = np.nan

    ece, mce = calculate_calibration_errors(
        y_true,
        pos_probs if probs.shape[1] == 2 else probs.max(axis=1),
        CFG["ECE_BINS"]
    )
    out[f"{prefix}ECE"] = ece
    out[f"{prefix}MCE"] = mce

    ent = predictive_entropy(probs)
    out[f"{prefix}Mean_Entropy"] = float(ent.mean())
    out[f"{prefix}Std_Entropy"] = float(ent.std())

    # Misclassification-as-OoD style score:
    # use entropy as uncertainty score, positives are incorrect predictions.
    miscls = (y_pred != y_true).astype(int)
    if miscls.sum() > 0 and miscls.sum() < len(miscls):
        out[f"{prefix}Miscls_AUROC"] = roc_auc_score(miscls, ent)
        out[f"{prefix}Miscls_AUPR"] = average_precision_score(miscls, ent)
        out[f"{prefix}Miscls_FPR95TPR"] = fpr_at_95_tpr(miscls, ent)
    else:
        out[f"{prefix}Miscls_AUROC"] = np.nan
        out[f"{prefix}Miscls_AUPR"] = np.nan
        out[f"{prefix}Miscls_FPR95TPR"] = np.nan

    return out

def plot_reliability(y_true, probs_pos, n_bins=10, title="Reliability Diagram"):
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    ids = np.digitize(probs_pos, bins) - 1
    ids = np.clip(ids, 0, n_bins - 1)

    accs, confs = [], []
    for b in range(n_bins):
        idx = ids == b
        if idx.sum() > 0:
            accs.append(y_true[idx].mean())
            confs.append(probs_pos[idx].mean())

    plt.figure(figsize=(5, 5))
    plt.plot([0, 1], [0, 1], "--")
    plt.plot(confs, accs, marker="o")
    plt.xlabel("Confidence")
    plt.ylabel("Empirical Accuracy")
    plt.title(title)
    plt.grid(True)
    plt.show()

def plot_joint_reliability(models_data, n_bins=10, title="Joint Reliability Diagram"):
    plt.figure(figsize=(8, 8))
    plt.plot([0, 1], [0, 1], "k--", label="Perfectly Calibrated")

    for model_name, y_true, probs_pos in models_data:
        y_true = np.asarray(y_true).astype(int)
        probs_pos = np.asarray(probs_pos).astype(float)

        bins = np.linspace(0.0, 1.0, n_bins + 1)
        bin_ids = np.digitize(probs_pos, bins) - 1
        bin_ids = np.clip(bin_ids, 0, n_bins - 1)

        accs, confs = [], []
        for b in range(n_bins):
            idx = bin_ids == b
            if idx.sum() > 0:
                accs.append(y_true[idx].mean())
                confs.append(probs_pos[idx].mean())
        if accs and confs:
            plt.plot(confs, accs, marker="o", label=model_name)

    plt.xlabel("Confidence")
    plt.ylabel("Empirical Accuracy")
    plt.title(title)
    plt.legend()
    plt.grid(True)
    plt.show()


In [ ]:
model = GraphEdgeModel(
    node_in_dim=data.x.size(1),
    edge_dim=data.edge_attr.size(1),
    hidden_dim=CFG["GNN_HIDDEN"],
    gnn_layers=CFG["GNN_LAYERS"],
    gnn_dropout=CFG["GNN_DROPOUT"],
    cls_hidden=CFG["CLS_HIDDEN"],
    cls_dropout=CFG["CLS_DROPOUT"],
    out_dim=n_classes
).to(device)

# Class weights from train labels
train_labels_t = data.edge_label[data.train_eid]
counts = torch.bincount(train_labels_t, minlength=n_classes).float()
class_weights = counts.sum() / (counts.clamp_min(1.0) * n_classes)
class_weights = class_weights.to(device)

criterion = nn.CrossEntropyLoss(
    weight=class_weights,
    label_smoothing=CFG["LABEL_SMOOTHING"]
)
optimizer = torch.optim.AdamW(model.parameters(), lr=CFG["LR"], weight_decay=CFG["WEIGHT_DECAY"])
scaler_amp = torch.cuda.amp.GradScaler(enabled=(CFG["USE_AMP"] and device.type == "cuda"))

best_state = None
best_val_f1 = -1
patience = 0

def eval_loader(model, loader, mc_dropout=False, T=1):
    if mc_dropout:
        model.train()
    else:
        model.eval()

    probs_all, y_all_list = [], []
    with torch.no_grad():
        if T == 1:
            for batch in loader:
                batch = batch.to(device)
                logits = model(batch)
                probs = F.softmax(logits, dim=1)
                probs_all.append(probs.cpu())
                y_all_list.append(batch.edge_label.cpu())
        else:
            # MC dropout predictive average
            per_pass = []
            true_ref = None
            for _ in range(T):
                pass_probs = []
                pass_true = []
                for batch in loader:
                    batch = batch.to(device)
                    logits = model(batch)
                    probs = F.softmax(logits, dim=1)
                    pass_probs.append(probs.cpu())
                    pass_true.append(batch.edge_label.cpu())
                per_pass.append(torch.cat(pass_probs, dim=0).unsqueeze(0))
                if true_ref is None:
                    true_ref = torch.cat(pass_true, dim=0)
            probs_all = [torch.cat(per_pass, dim=0).mean(dim=0)]
            y_all_list = [true_ref]

    probs_np = torch.cat(probs_all, dim=0).numpy()
    y_np = torch.cat(y_all_list, dim=0).numpy()
    return y_np, probs_np

history = {"train_loss": [], "val_f1": [], "val_acc": []}

for epoch in range(1, CFG["EPOCHS"] + 1):
    model.train()
    total_loss = 0.0
    total_items = 0

    for batch in train_loader:
        batch = batch.to(device)
        optimizer.zero_grad(set_to_none=True)

        with torch.cuda.amp.autocast(enabled=(CFG["USE_AMP"] and device.type == "cuda")):
            logits = model(batch)
            loss = criterion(logits, batch.edge_label)

        scaler_amp.scale(loss).backward()
        scaler_amp.step(optimizer)
        scaler_amp.update()

        total_loss += loss.item() * batch.edge_label.size(0)
        total_items += batch.edge_label.size(0)

    train_loss = total_loss / max(total_items, 1)
    y_val, p_val = eval_loader(model, val_loader, mc_dropout=False, T=1)
    val_stats = summarize_predictions(y_val, p_val)

    history["train_loss"].append(train_loss)
    history["val_f1"].append(val_stats["F1"])
    history["val_acc"].append(val_stats["Accuracy"])

    print(
        f"Epoch {epoch:03d} | "
        f"train_loss={train_loss:.4f} | "
        f"val_acc={val_stats['Accuracy']:.4f} | "
        f"val_f1={val_stats['F1']:.4f} | "
        f"val_ece={val_stats['ECE']:.4f}"
    )

    if val_stats["F1"] > best_val_f1:
        best_val_f1 = val_stats["F1"]
        best_state = copy.deepcopy(model.state_dict())
        patience = 0
    else:
        patience += 1
        if patience >= CFG["PATIENCE"]:
            print("Early stopping triggered.")
            break

model.load_state_dict(best_state)
print("Loaded best validation model.")

In [ ]:
# ============================================================
# 14) BASE & TEMP SCALED TEST RESULTS
# ============================================================

# Helper to evaluate loaders with timing and memory tracking
def eval_and_track_loader(model_instance, loader, mc_dropout=False, T=1, model_type=""):
    torch.cuda.empty_cache()
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()

    start_time = time.time()
    y_true_np, probs_np = eval_loader(model_instance, loader, mc_dropout, T)
    end_time = time.time()

    runtime = end_time - start_time
    memory = torch.cuda.max_memory_allocated() if torch.cuda.is_available() else 0

    print(f"{model_type} runtime: {runtime:.2f}s")
    print(f"{model_type} peak GPU memory: {memory / (1024**2):.2f} MB") # Convert to MB

    return y_true_np, probs_np, runtime, memory

# 1. Evaluate Base Model
y_test_base, p_test_base, runtime_base, memory_base = eval_and_track_loader(model, test_loader, mc_dropout=False, T=1, model_type="Base Model")
base_stats = summarize_predictions(y_test_base, p_test_base, prefix="Base_")
base_stats["Base_Runtime_s"] = runtime_base
base_stats["Base_Memory_MB"] = memory_base / (1024**2) # Store in MB
print("Base Model evaluated.")

# 2. Fit Temperature Scaling
print("\nFitting Temperature Scaling...")
temp_model = ModelWithTemperature(model)
temp_model.set_temperature(val_loader, device)

# 3. Evaluate Scaled Model
# Helper to eval temp model specifically
def eval_temp_loader(t_model, loader):
    t_model.eval()
    probs_all, y_all_list = [], []
    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)
            # t_model forward calls temperature_scale(model(batch))
            logits = t_model(batch)
            probs = F.softmax(logits, dim=1)
            probs_all.append(probs.cpu())
            y_all_list.append(batch.edge_label.cpu())
    return torch.cat(y_all_list).numpy(), torch.cat(probs_all, dim=0).numpy()

y_test_temp, p_test_temp, runtime_temp, memory_temp = eval_and_track_loader(temp_model, test_loader, model_type="Temp Scaled Model")
temp_stats = summarize_predictions(y_test_temp, p_test_temp, prefix="Temp_")
temp_stats["Temp_Runtime_s"] = runtime_temp
temp_stats["Temp_Memory_MB"] = memory_temp / (1024**2) # Store in MB
print("Temperature Scaled Model evaluated.")

display(pd.DataFrame([base_stats, temp_stats]).T)

In [ ]:

# ============================================================
# 15) MC DROPOUT RESULTS
# ============================================================
y_test_mc, p_test_mc, runtime_mc, memory_mc = eval_and_track_loader(model, test_loader, mc_dropout=True, T=CFG["MC_PASSES"], model_type="MC Dropout Model")
mc_stats = summarize_predictions(y_test_mc, p_test_mc, prefix="MC_")
mc_stats["MC_Runtime_s"] = runtime_mc
mc_stats["MC_Memory_MB"] = memory_mc / (1024**2) # Store in MB
pd.DataFrame([mc_stats]).T

In [ ]:

# ============================================================
# 16) TRAINING CURVES
# ============================================================
plt.figure(figsize=(12, 4))
plt.subplot(1, 3, 1)
plt.plot(history["train_loss"])
plt.title("Train Loss")
plt.grid(True)

plt.subplot(1, 3, 2)
plt.plot(history["val_acc"])
plt.title("Validation Accuracy")
plt.grid(True)

plt.subplot(1, 3, 3)
plt.plot(history["val_f1"])
plt.title("Validation F1")
plt.grid(True)
plt.show()


In [ ]:

# ============================================================
# 17) RELIABILITY DIAGRAMS (BASE VS MC DROPOUT)
# ============================================================
plot_reliability(y_test_base, p_test_base[:, 1], n_bins=CFG["ECE_BINS"], title="Base GNN Reliability")
plot_reliability(y_test_mc,   p_test_mc[:, 1],   n_bins=CFG["ECE_BINS"], title="MC Dropout Reliability")


In [ ]:

# ============================================================
# 18) EXTRACT FROZEN EDGE EMBEDDINGS
# ============================================================
model.eval()
with torch.no_grad():
    node_emb = model.full_node_embeddings(data, device=device)  # [N, hidden]

def build_edge_embeddings(eids_tensor):
    eids = eids_tensor.cpu().numpy()
    src_e = src_id[eids]
    dst_e = dst_id[eids]

    src_emb = node_emb[src_e]
    dst_emb = node_emb[dst_e]
    eattr = data.edge_attr[eids].cpu()

    z = torch.cat([src_emb, dst_emb, eattr], dim=1)
    y = data.edge_label[eids].cpu()
    return z.float(), y.long(), eids

Z_train_full, y_train_full, eids_train_full = build_edge_embeddings(data.train_eid)
Z_val_full,   y_val_full,   eids_val_full   = build_edge_embeddings(data.val_eid)
Z_test_full,  y_test_full2, eids_test_full  = build_edge_embeddings(data.test_eid)

print("Z_train_full:", tuple(Z_train_full.shape))
print("Z_val_full  :", tuple(Z_val_full.shape))
print("Z_test_full :", tuple(Z_test_full.shape))


In [ ]:

# ============================================================
# 19) PREPARE NUTS SUBSET
# ============================================================
# NUTS must be run on a manageable subset in Colab.
# We sample a stratified subset from TRAIN edge embeddings.

nuts_subset = min(CFG["NUTS_SUBSET_EDGES"], len(y_train_full))
subset_idx = train_test_split(
    np.arange(len(y_train_full)),
    train_size=nuts_subset,
    random_state=CFG["SEED"],
    stratify=y_train_full.numpy()
)[0]

Z_nuts = Z_train_full[subset_idx].to(device)
y_nuts = y_train_full[subset_idx].to(device)

# Standardize embedding features using training subset statistics only
z_mean = Z_nuts.mean(dim=0, keepdim=True)
z_std = Z_nuts.std(dim=0, keepdim=True).clamp_min(1e-6)

def standardize_z(z):
    return (z - z_mean) / z_std

Z_nuts_std = standardize_z(Z_nuts)
Z_val_std = standardize_z(Z_val_full.to(device))
Z_test_std = standardize_z(Z_test_full.to(device))

print("NUTS train subset:", tuple(Z_nuts_std.shape))


In [ ]:

# ============================================================
# 20) DETERMINE A PRACTICAL BAYESIAN CLASSIFIER SIZE
# ============================================================
# NUTS cost grows quickly with parameter dimension.
# To keep Colab practicality, we use a smaller hidden layer than the full
# deterministic classifier if needed.

BAYES_HIDDEN = min(64, CFG["CLS_HIDDEN"])
ZDIM = Z_nuts_std.size(1)
print("Bayesian classifier input dim:", ZDIM)
print("Bayesian hidden dim:", BAYES_HIDDEN)


In [ ]:

# ============================================================
# 21) PYRO MODEL FOR NUTS
# ============================================================
# Bayesian MLP over frozen edge embeddings.
# Binary case uses categorical logits with 2 outputs.

def bayes_edge_mlp(z, y=None):
    in_dim = z.size(1)
    hid = BAYES_HIDDEN
    out_dim = n_classes

    # Priors
    w1 = pyro.sample("w1", dist.Normal(
        torch.zeros(in_dim, hid, device=z.device),
        torch.ones(in_dim, hid, device=z.device)
    ).to_event(2))
    b1 = pyro.sample("b1", dist.Normal(
        torch.zeros(hid, device=z.device),
        torch.ones(hid, device=z.device)
    ).to_event(1))

    w2 = pyro.sample("w2", dist.Normal(
        torch.zeros(hid, out_dim, device=z.device),
        torch.ones(hid, out_dim, device=z.device)
    ).to_event(2))
    b2 = pyro.sample("b2", dist.Normal(
        torch.zeros(out_dim, device=z.device),
        torch.ones(out_dim, device=z.device)
    ).to_event(1))

    h = torch.tanh(z @ w1 + b1)
    logits = h @ w2 + b2

    with pyro.plate("data", z.size(0)):
        pyro.sample("obs", dist.Categorical(logits=logits), obs=y)

    return logits


In [ ]:

# ============================================================
# 22) RUN NUTS
# ============================================================
pyro.clear_param_store()
seed_everything(CFG["SEED"])

nuts_kernel = NUTS(
    bayes_edge_mlp,
    adapt_step_size=True,
    adapt_mass_matrix=True,
    target_accept_prob=CFG["NUTS_TARGET_ACCEPT"],
    max_tree_depth=CFG["NUTS_MAX_TREE_DEPTH"],
    jit_compile=CFG["NUTS_JIT_COMPILE"]
)

mcmc = MCMC(
    nuts_kernel,
    num_samples=CFG["NUTS_NUM_SAMPLES"],
    warmup_steps=CFG["NUTS_NUM_WARMUP"],
    num_chains=CFG["NUTS_NUM_CHAINS"],
    disable_progbar=False
)

t0 = time.time()
mcmc.run(Z_nuts_std, y_nuts)
nuts_runtime = time.time() - t0
print(f"NUTS runtime: {nuts_runtime/60:.2f} minutes")

nuts_samples = mcmc.get_samples()
for k, v in nuts_samples.items():
    print(k, tuple(v.shape))


In [ ]:

# ============================================================
# 23) NUTS DIAGNOSTICS
# ============================================================
print("\nPyro summary:")
mcmc.summary()

diag = mcmc.diagnostics()
print("\nDiagnostics keys:", diag.keys())

# Convert to ArviZ for richer diagnostics if possible
try:
    idata = az.from_pyro(mcmc)
    display(az.summary(idata, round_to=4))
    az.plot_trace(idata, var_names=["b2"])
    plt.show()
except Exception as e:
    print("ArviZ conversion skipped:", e)


In [ ]:

# ============================================================
# 24) POSTERIOR PREDICTIVE FOR VAL / TEST
# ============================================================
predictive = Predictive(
    bayes_edge_mlp,
    posterior_samples=nuts_samples,
    return_sites=("obs", "_RETURN")
)

# Helper to track NUTS predictive runtime and memory
def nuts_predict_and_track(predictive_obj, Z_val_std, Z_test_std, model_type=""):
    torch.cuda.empty_cache()
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()

    start_time = time.time()
    with torch.no_grad():
        post_val = predictive_obj(Z_val_std)
        post_test = predictive_obj(Z_test_std)
    end_time = time.time()

    runtime = end_time - start_time
    memory = torch.cuda.max_memory_allocated() if torch.cuda.is_available() else 0

    print(f"{model_type} runtime: {runtime:.2f}s")
    print(f"{model_type} peak GPU memory: {memory / (1024**2):.2f} MB") # Convert to MB

    return post_val, post_test, runtime, memory

logits_val_samples, logits_test_samples, runtime_nuts, memory_nuts = nuts_predict_and_track(predictive, Z_val_std, Z_test_std, model_type="NUTS Model")

probs_val_samples = torch.softmax(logits_val_samples["_RETURN"], dim=-1)
probs_test_samples = torch.softmax(logits_test_samples["_RETURN"], dim=-1)

p_val_nuts = probs_val_samples.mean(dim=0).cpu().numpy()
p_test_nuts = probs_test_samples.mean(dim=0).cpu().numpy()

y_val_nuts = y_val_full.numpy()
y_test_nuts = y_test_full2.numpy()

nuts_stats = summarize_predictions(y_test_nuts, p_test_nuts, prefix="NUTS_")
nuts_stats["NUTS_Runtime_s"] = runtime_nuts
nuts_stats["NUTS_Memory_MB"] = memory_nuts / (1024**2) # Store in MB
pd.DataFrame([nuts_stats]).T


In [ ]:

# ============================================================
# 25) SAMPLING-EVOLUTION ANALYSIS FOR NUTS
# ============================================================
# Instead of Gibbs iteration traces, we examine cumulative posterior-sample
# aggregation on the validation set.

cum_metrics = {
    "samples_used": [],
    "val_acc": [],
    "val_f1": [],
    "val_ece": [],
    "val_mce": [],
    "val_auroc": [],
}

for s in range(1, probs_val_samples.size(0) + 1):
    p_cum = probs_val_samples[:s].mean(dim=0).cpu().numpy()
    stats_cum = summarize_predictions(y_val_nuts, p_cum)

    cum_metrics["samples_used"].append(s)
    cum_metrics["val_acc"].append(stats_cum["Accuracy"])
    cum_metrics["val_f1"].append(stats_cum["F1"])
    cum_metrics["val_ece"].append(stats_cum["ECE"])
    cum_metrics["val_mce"].append(stats_cum["MCE"])
    cum_metrics["val_auroc"].append(stats_cum["ROC_AUC"])

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.ravel()

axes[0].plot(cum_metrics["samples_used"], cum_metrics["val_acc"])
axes[0].set_title("Validation Accuracy vs Retained NUTS Samples")
axes[0].grid(True)

axes[1].plot(cum_metrics["samples_used"], cum_metrics["val_f1"])
axes[1].set_title("Validation F1 vs Retained NUTS Samples")
axes[1].grid(True)

axes[2].plot(cum_metrics["samples_used"], cum_metrics["val_ece"])
axes[2].set_title("Validation ECE vs Retained NUTS Samples")
axes[2].grid(True)

axes[3].plot(cum_metrics["samples_used"], cum_metrics["val_auroc"])
axes[3].set_title("Validation ROC-AUC vs Retained NUTS Samples")
axes[3].grid(True)

plt.tight_layout()
plt.show()


In [ ]:
# ============================================================
# 26) COMPARE BASE / TEMP / MC DROPOUT / NUTS
# ============================================================
rows = []

# Base
rows.append({
    "Model": "Base GNN",
    "Accuracy": base_stats["Base_Accuracy"],
    "F1": base_stats["Base_F1"],
    "ROC_AUC": base_stats["Base_ROC_AUC"],
    "ECE": base_stats["Base_ECE"],
    "MCE": base_stats["Base_MCE"],
    "Mean Entropy": base_stats["Base_Mean_Entropy"],
    "Miscls AUROC": base_stats["Base_Miscls_AUROC"],
    "Runtime_s": base_stats["Base_Runtime_s"],
    "Memory_MB": base_stats["Base_Memory_MB"],
})

# Temp Scaled
rows.append({
    "Model": "GNN + Temp Scaling",
    "Accuracy": temp_stats["Temp_Accuracy"],
    "F1": temp_stats["Temp_F1"],
    "ROC_AUC": temp_stats["Temp_ROC_AUC"],
    "ECE": temp_stats["Temp_ECE"],
    "MCE": temp_stats["Temp_MCE"],
    "Mean Entropy": temp_stats["Temp_Mean_Entropy"],
    "Miscls AUROC": temp_stats["Temp_Miscls_AUROC"],
    "Runtime_s": temp_stats["Temp_Runtime_s"],
    "Memory_MB": temp_stats["Temp_Memory_MB"],
})

# MC Dropout
rows.append({
    "Model": "GNN + MC Dropout",
    "Accuracy": mc_stats["MC_Accuracy"],
    "F1": mc_stats["MC_F1"],
    "ROC_AUC": mc_stats["MC_ROC_AUC"],
    "ECE": mc_stats["MC_ECE"],
    "MCE": mc_stats["MC_MCE"],
    "Mean Entropy": mc_stats["MC_Mean_Entropy"],
    "Miscls AUROC": mc_stats["MC_Miscls_AUROC"],
    "Runtime_s": mc_stats["MC_Runtime_s"],
    "Memory_MB": mc_stats["MC_Memory_MB"],
})

# NUTS
rows.append({
    "Model": "GNN + MC Dropout + NUTS",
    "Accuracy": nuts_stats["NUTS_Accuracy"],
    "F1": nuts_stats["NUTS_F1"],
    "ROC_AUC": nuts_stats["NUTS_ROC_AUC"],
    "ECE": nuts_stats["NUTS_ECE"],
    "MCE": nuts_stats["NUTS_MCE"],
    "Mean Entropy": nuts_stats["NUTS_Mean_Entropy"],
    "Miscls AUROC": nuts_stats["NUTS_Miscls_AUROC"],
    "Runtime_s": nuts_stats["NUTS_Runtime_s"],
    "Memory_MB": nuts_stats["NUTS_Memory_MB"],
})

comparison = pd.DataFrame(rows)
comparison_columns = [
    "Model", "Accuracy", "F1", "ROC_AUC", "ECE", "MCE",
    "Mean Entropy", "Miscls AUROC", "Runtime_s", "Memory_MB"
]
comparison = comparison[comparison_columns]
display(comparison.round(4))

In [ ]:

# ============================================================
# 27) CALIBRATION PLOTS
# ============================================================

models_for_joint_plot = [
    ("Base GNN", y_test_base, p_test_base[:, 1]),
    ("GNN + Temp Scaling", y_test_temp, p_test_temp[:, 1]),
    ("MC Dropout", y_test_mc, p_test_mc[:, 1]),
    ("NUTS", y_test_nuts, p_test_nuts[:, 1])
]

plot_joint_reliability(models_for_joint_plot, n_bins=CFG["ECE_BINS"], title="Joint Reliability Diagram for All Models")


In [ ]:

# ============================================================
# 28) CALIBRATION METRICS BAR CHART
# ============================================================
plt.figure(figsize=(8, 4))
x = np.arange(3)
ece_vals = [
    base_stats["Base_ECE"],
    mc_stats["MC_ECE"],
    nuts_stats["NUTS_ECE"]
]
mce_vals = [
    base_stats["Base_MCE"],
    mc_stats["MC_MCE"],
    nuts_stats["NUTS_MCE"]
]

w = 0.35
plt.bar(x - w/2, ece_vals, width=w, label="ECE")
plt.bar(x + w/2, mce_vals, width=w, label="MCE")
plt.xticks(x, ["Base GNN", "MC Dropout", "NUTS"])
plt.title("Calibration Comparison")
plt.legend()
plt.grid(True, axis="y")
plt.show()


In [ ]:

# ============================================================
# 29) CONFUSION MATRICES
# ============================================================
def plot_cm(y_true, probs, title):
    y_pred = probs.argmax(axis=1)
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(4, 4))
    plt.imshow(cm, cmap="Blues")
    plt.title(title)
    plt.xlabel("Predicted")
    plt.ylabel("True")
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            plt.text(j, i, int(cm[i, j]), ha="center", va="center")
    plt.colorbar()
    plt.show()

plot_cm(y_test_base, p_test_base, "Base GNN")
plot_cm(y_test_mc, p_test_mc, "GNN + MC Dropout")
plot_cm(y_test_nuts, p_test_nuts, "GNN + MC Dropout + NUTS")


In [ ]:

# ============================================================
# 30) ENTROPY / CONFIDENCE DISTRIBUTIONS
# ============================================================
entropy_base = predictive_entropy(p_test_base)
entropy_mc   = predictive_entropy(p_test_mc)
entropy_nuts = predictive_entropy(p_test_nuts)

conf_base = p_test_base.max(axis=1)
conf_mc   = p_test_mc.max(axis=1)
conf_nuts = p_test_nuts.max(axis=1)

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.hist(entropy_base, bins=40, alpha=0.5, label="Base")
plt.hist(entropy_mc, bins=40, alpha=0.5, label="MC")
plt.hist(entropy_nuts, bins=40, alpha=0.5, label="NUTS")
plt.title("Predictive Entropy Distribution")
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
plt.hist(conf_base, bins=40, alpha=0.5, label="Base")
plt.hist(conf_mc, bins=40, alpha=0.5, label="MC")
plt.hist(conf_nuts, bins=40, alpha=0.5, label="NUTS")
plt.title("Prediction Confidence Distribution")
plt.legend()
plt.grid(True)
plt.show()


In [ ]:

# ============================================================
# 31) ROC + PR FOR MISCLASSIFICATION DETECTION
# ============================================================
def plot_miscls_curves(y_true, probs, label):
    y_pred = probs.argmax(axis=1)
    miscls = (y_pred != y_true).astype(int)
    score = predictive_entropy(probs)

    if miscls.sum() == 0 or miscls.sum() == len(miscls):
        return None

    fpr, tpr, _ = roc_curve(miscls, score)
    prec, rec, _ = precision_recall_curve(miscls, score)
    return fpr, tpr, prec, rec

curves = {
    "Base GNN": plot_miscls_curves(y_test_base, p_test_base, "Base"),
    "MC Dropout": plot_miscls_curves(y_test_mc, p_test_mc, "MC"),
    "NUTS": plot_miscls_curves(y_test_nuts, p_test_nuts, "NUTS")
}

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
for name, vals in curves.items():
    if vals is not None:
        fpr, tpr, _, _ = vals
        plt.plot(fpr, tpr, label=name)
plt.plot([0, 1], [0, 1], "--")
plt.title("Misclassification Detection ROC")
plt.xlabel("FPR")
plt.ylabel("TPR")
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
for name, vals in curves.items():
    if vals is not None:
        _, _, prec, rec = vals
        plt.plot(rec, prec, label=name)
plt.title("Misclassification Detection PR")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.legend()
plt.grid(True)
plt.show()


In [ ]:

# ============================================================
# 33) SAVE RESULTS
# ============================================================
results_dir = "/content/nuts_gibbon_results"
os.makedirs(results_dir, exist_ok=True)

comparison.to_csv(os.path.join(results_dir, "comparison_metrics.csv"), index=False)

with open(os.path.join(results_dir, "config.json"), "w") as f:
    json.dump(CFG, f, indent=2)

torch.save(best_state, os.path.join(results_dir, "best_base_model.pt"))
torch.save({
    "z_mean": z_mean.cpu(),
    "z_std": z_std.cpu(),
    "nuts_samples": {k: v.cpu() for k, v in nuts_samples.items()}
}, os.path.join(results_dir, "nuts_posterior.pt"))

print("Saved artifacts to:", results_dir)
print("Files:", os.listdir(results_dir))


## Runtime and Memory Comparison

In [ ]:
# ============================================================
# 33.1) RUNTIME COMPARISON
# ============================================================
import seaborn as sns

models = comparison["Model"]
runtimes = comparison["Runtime_s"]

plt.figure(figsize=(10, 6))
sns.barplot(x=models, y=runtimes, palette="viridis")
plt.title("Model Inference Runtime Comparison (Test Set)")
plt.xlabel("Model")
plt.ylabel("Runtime (seconds)")
plt.xticks(rotation=45, ha='right')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 33.2) MEMORY USAGE COMPARISON
# ============================================================
models = comparison["Model"]
memory_usage = comparison["Memory_MB"]

plt.figure(figsize=(10, 6))
sns.barplot(x=models, y=memory_usage, palette="magma")
plt.title("Model Peak GPU Memory Usage Comparison (Test Set)")
plt.xlabel("Model")
plt.ylabel("Peak GPU Memory (MB)")
plt.xticks(rotation=45, ha='right')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

In [ ]:

# ============================================================
# 32) UMAP VISUALISATION OF NODE EMBEDDINGS
# ============================================================
# This is mainly qualitative and can be expensive on very large node sets.
# We therefore sample a subset of nodes if needed.

node_sample_n = min(15000, node_emb.size(0))
node_idx = np.random.RandomState(CFG["SEED"]).choice(node_emb.size(0), size=node_sample_n, replace=False)
node_emb_sample = node_emb[node_idx].numpy()

# Approximate a node label from incident training edges (majority label)
node_incident_label = np.zeros(node_emb.size(0), dtype=np.int64)
node_counts = np.zeros(node_emb.size(0), dtype=np.int64)

train_edges = data.train_eid.cpu().numpy()
for eid in train_edges[: min(len(train_edges), 500000)]:
    s = src_id[eid]
    d = dst_id[eid]
    lab = y_all[eid]
    node_incident_label[s] += lab
    node_incident_label[d] += lab
    node_counts[s] += 1
    node_counts[d] += 1

node_majority = (node_incident_label / np.maximum(node_counts, 1) >= 0.5).astype(int)
node_label_sample = node_majority[node_idx]

reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=CFG["SEED"])
proj = reducer.fit_transform(node_emb_sample)

plt.figure(figsize=(7, 6))
plt.scatter(proj[:, 0], proj[:, 1], c=node_label_sample, s=5, alpha=0.7)
plt.title("UMAP Projection of Learned Node Embeddings")
plt.show()


In [ ]:
# ============================================================
# 34) SAVE RESULTS
# ============================================================
results_dir = "/content/nuts_gibbon_results_new"
os.makedirs(results_dir, exist_ok=True)

comparison.to_csv(os.path.join(results_dir, "comparison_metrics.csv"), index=False)

with open(os.path.join(results_dir, "config.json"), "w") as f:
    json.dump(CFG, f, indent=2)

torch.save(best_state, os.path.join(results_dir, "best_base_model.pt"))
torch.save({
    "z_mean": z_mean.cpu(),
    "z_std": z_std.cpu(),
    "nuts_samples": {k: v.cpu() for k, v in nuts_samples.items()}
}, os.path.join(results_dir, "nuts_posterior.pt"))

print("Saved artifacts to:", results_dir)
print("Files:", os.listdir(results_dir))



---



---



---



---



---



---



---



---



---





---



---



---



---



---



---



---



---



---



---



---





---



---



---



---



---



---



---



---



---



---



---

